In [3]:
import os
import cv2
import numpy as np
import easyocr

# 1. 모델 불러오기
'''
def __init__(self, lang_list, gpu=True, model_storage_directory=None,
                 user_network_directory=None, detect_network="craft", 
                 recog_network='standard', download_enabled=True, 
                 detector=True, recognizer=True, verbose=True, 
                 quantize=True, cudnn_benchmark=False):
'''
reader = easyocr.Reader(['ja'])

# 2. 이미지 경로 지정 및 추론
image_path = '25.png'
result = reader.readtext(image_path)

def bbox_to_rect(bbox):
    """bbox(4점) -> (x_min, y_min, x_max, y_max)"""
    xs = [int(p[0]) for p in bbox]
    ys = [int(p[1]) for p in bbox]
    return min(xs), min(ys), max(xs), max(ys)

# 3. detection / recognition 분리 출력
print("=" * 70)
print(f"총 {len(result)}개의 영역(Detection)에서 텍스트(Recognition)를 인식했습니다.")
print("=" * 70)

# (A) Detection 결과: 영역 좌표만
print("\n[A] Detection 영역 좌표(박스)\n" + "-" * 70)
detections = []
for idx, (bbox, _text, _conf) in enumerate(result, 1):
    x_min, y_min, x_max, y_max = bbox_to_rect(bbox)
    detections.append((idx, bbox, (x_min, y_min, x_max, y_max)))

for idx, bbox, (x_min, y_min, x_max, y_max) in detections:
    # 보기 쉽게: 사각 범위 + 원본 4점 좌표
    print(f"[{idx}] rect: ({x_min}, {y_min}) ~ ({x_max}, {y_max})")
    print(f"    points: {[(int(p[0]), int(p[1])) for p in bbox]}")

# (B) Recognition 결과: 텍스트/신뢰도만
print("\n[B] Recognition 결과(각 Detection 영역의 텍스트)\n" + "-" * 70)
recognitions = []
for idx, (_bbox, text, confidence) in enumerate(result, 1):
    recognitions.append((idx, text, float(confidence)))

for idx, text, confidence in recognitions:
    print(f"[{idx}] text: {text}")
    print(f"    confidence: {confidence:.2%}")

Neither CUDA nor MPS are available - defaulting to CPU. Note: This module is much faster with a GPU.
c:\Users\a\Desktop\수업 자료\개인공부\project\my_easyocr\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


총 13개의 영역(Detection)에서 텍스트(Recognition)를 인식했습니다.

[A] Detection 영역 좌표(박스)
----------------------------------------------------------------------
[1] rect: (75, 77) ~ (113, 147)
    points: [(75, 77), (113, 77), (113, 147), (75, 147)]
[2] rect: (458, 54) ~ (490, 122)
    points: [(458, 54), (490, 54), (490, 122), (458, 122)]
[3] rect: (459, 125) ~ (487, 190)
    points: [(459, 125), (487, 125), (487, 190), (459, 190)]
[4] rect: (76, 148) ~ (102, 234)
    points: [(76, 148), (102, 148), (102, 234), (76, 234)]
[5] rect: (484, 248) ~ (514, 256)
    points: [(484, 248), (514, 248), (514, 256), (484, 256)]
[6] rect: (433, 255) ~ (517, 275)
    points: [(433, 255), (517, 255), (517, 275), (433, 275)]
[7] rect: (64, 328) ~ (112, 400)
    points: [(64, 328), (112, 328), (112, 400), (64, 400)]
[8] rect: (89, 381) ~ (109, 417)
    points: [(89, 381), (109, 381), (109, 417), (89, 417)]
[9] rect: (414, 520) ~ (486, 708)
    points: [(414, 520), (486, 520), (486, 708), (414, 708)]
[10] rect: (505, 6

In [4]:
# 4. Detection 박스를 이미지에 그려 저장
out_path = None

# 이미지 읽기
drawn = cv2.imread(image_path)
if drawn is None:
    raise FileNotFoundError(f"이미지를 불러올 수 없습니다: {image_path}")

# 박스 그리기 (빨간색) + 박스 번호 라벨
for idx, (bbox, text, conf) in enumerate(result, 1):
    pts = np.array(bbox, dtype=np.int32)
    cv2.polylines(drawn, [pts], isClosed=True, color=(0, 0, 255), thickness=2)
    # 박스 번호를 좌측 상단 근처에 표시
    label_pos = (int(pts[0][0]), int(pts[0][1]) - 6)
    cv2.putText(drawn, f"[{idx}]", label_pos,
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2, cv2.LINE_AA)

base, ext = os.path.splitext(image_path)
out_path = f"{base}_box{ext}"

ok = cv2.imwrite(out_path, drawn)
if not ok:
    raise RuntimeError(f"이미지 저장에 실패했습니다: {out_path}")

print(f"Detection 박스를 그린 이미지를 저장했습니다: {out_path}")

Detection 박스를 그린 이미지를 저장했습니다: 25_box.png
